# ComplianceGuard — GRPO Training (T4-safe, no oracle)

**Stack:** Unsloth + TRL GRPO | **Model:** Qwen2.5-1.5B-Instruct 4-bit QLoRA  
**Runtime:** T4 GPU (free tier works)

### Why this notebook is different from naive oracle approaches
- **Broken pattern:** model outputs 1 action → oracle completes episode → reward=0.999 always → reward_std=0.000 → model learns nothing
- **This notebook:** model outputs ALL actions in one completion → we run them for real → reward varies 0.05–1.0 → genuine learning signal

**Health check after step 5:** `reward_std` must be > 0.05. If it is 0.000, training is broken.

## Cell 1 — Install dependencies

In [ ]:
!pip install unsloth --quiet
!pip install 'trl>=0.15.0' datasets accelerate peft --quiet

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU'
vram = f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB' if torch.cuda.is_available() else ''
print(f'GPU: {gpu}  {vram}')
if not torch.cuda.is_available():
    raise RuntimeError('Switch to GPU: Runtime > Change runtime type > T4 GPU')

## Cell 2 — ComplianceGuard environment (inline, no server)

In [ ]:
import json, random

_FIRST = ['Alice','Bob','Carol','David','Emma','Frank','Grace','Henry','Iris','Jack']
_LAST  = ['Johnson','Martinez','White','Brown','Davis','Wilson','Anderson','Taylor']
_DOMS  = ['gmail.com','yahoo.com','company.com','corp.io','work.net']

def _name():
    f, l = random.choice(_FIRST), random.choice(_LAST)
    return f'{f} {l}', f, l

def _email(f, l): return f'{f.lower()}.{l.lower()}@{random.choice(_DOMS)}'

def _phone():
    a,m,e = random.randint(200,999), random.randint(100,999), random.randint(1000,9999)
    return random.choice([f'({a}) {m}-{e}', f'{a}.{m}.{e}', f'+1-{a}-{m}-{e}'])

def _obf_email(email):
    u,d = email.split('@'); p = d.split('.')
    return f'{u} [at] {p[0]} [dot] {p[1]}'

def _obf_phone(phone):
    return phone.replace('-',' dash ').replace('.',' dot ').replace('(','').replace(')','')

def gen_l1():
    nm,f,l = _name(); email,phone = _email(f,l), _phone()
    c = (f'2026-01-01 INFO user {nm} logged in from 10.0.0.5\n'
         f'2026-01-01 INFO support contact: {email}\n'
         f'2026-01-02 WARN call us at {phone} for help\n')
    return {'server_logs.txt': c}, [nm, email, phone]

def gen_l2():
    pp=[_name() for _ in range(3)]
    names=[p[0] for p in pp]; emails=[_email(p[1],p[2]) for p in pp]; phones=[_phone() for _ in range(3)]
    logs  = (f'2026-01-01 ERROR user {names[0]} failed login\n'
             f'2026-01-01 INFO support: {emails[0]}\n'
             f'2026-01-02 INFO {names[1]} called {phones[0]}\n')
    report= (f'Incident report\nContact: {names[2]}\n'
             f'Email: {emails[1]}\nPhone: {phones[1]}\n'
             f'Alt email: {emails[2]}\nAlt phone: {phones[2]}\n')
    return {'server_logs.txt': logs, 'incident_report.txt': report}, names+emails+phones

def gen_l3():
    pp=[_name() for _ in range(3)]
    names=[p[0] for p in pp]; re2=[_email(p[1],p[2]) for p in pp]; rp=[_phone() for _ in range(3)]
    oe=[_obf_email(e) for e in re2]; op=[_obf_phone(p) for p in rp]
    logs  = f'2026-01-01 INFO contact {names[0]} via {oe[0]}\n2026-01-02 WARN reach {names[1]} at {op[0]}\n'
    report= f'User: {names[2]}\nReach at: {oe[1]}\nPhone: {op[1]}\n'
    notes = f'Backup contacts: {oe[2]} / {op[2]}\nServer IP: 192.168.1.1\n'
    return {'server_logs.txt': logs, 'incident_report.txt': report, 'notes.txt': notes}, names+oe+op

def gen_l4():
    files, pii = gen_l3()
    rh = ['test@example.com','noreply@system.local','admin@localhost','000.000.0000']
    fn = list(files.keys())
    files[fn[0]] += f'System alert sent to {rh[0]}\nAuto-reply from {rh[1]}\n'
    files['system_log.txt'] = f'Automated notification: {rh[2]}\nTest phone: {rh[3]}\n'
    return files, pii

GENERATORS = {1: gen_l1, 2: gen_l2, 3: gen_l3, 4: gen_l4}
PHASE_TOOLS = {
    'SCAN':     ['list_files','read_file','flag_candidate','advance_phase'],
    'CLASSIFY': ['list_candidates','classify_candidate','advance_phase'],
    'REDACT':   ['redact_span','submit'],
}


class ComplianceGuardEnv:
    def __init__(self): self.reset()

    def reset(self, seed=None, level=1, **kw):
        if seed is not None: random.seed(seed)
        self.level = level
        self.virtual_fs, self.pii_list = GENERATORS.get(level, gen_l1)()
        self.candidates = {}; self._cid = 0; self.phase = 'SCAN'
        self.done = False; self.reward = 0.0; self.cumulative = 0.0
        return f'Reset L{level}. Files: {list(self.virtual_fs.keys())}. Use list_files -> read_file -> flag_candidate -> advance_phase.'

    def step(self, action_json):
        try: parsed = json.loads(action_json); tool = parsed.get('tool','')
        except: return self._obs(-0.05,'Error: send valid JSON with tool field.')
        fn = getattr(self, f'_t_{tool}', None)
        if fn is None: return self._obs(-0.05, f'Unknown tool {tool!r}. Allowed: {PHASE_TOOLS[self.phase]}')
        try: result = fn(parsed)
        except ValueError as e: return self._obs(-0.05, str(e))
        if self.done: return self._obs(self.reward, result, done=True)
        return self._obs(0.0, result)

    def _req(self, phase, tool):
        if self.phase != phase: raise ValueError(f'{tool!r} only in {phase}. Currently {self.phase}.')

    def _t_list_files(self, p):
        self._req('SCAN','list_files'); return f'Files: {list(self.virtual_fs.keys())}'

    def _t_read_file(self, p):
        self._req('SCAN','read_file'); fp=p.get('file_path','')
        if fp not in self.virtual_fs: raise ValueError(f'{fp!r} not found.')
        return f'=== {fp} ===\n{self.virtual_fs[fp]}'

    def _t_flag_candidate(self, p):
        self._req('SCAN','flag_candidate'); text=p.get('text','').strip()
        if not text: raise ValueError('text required.')
        cid=f'c{self._cid}'; self._cid+=1
        self.candidates[cid]={'text':text,'file_path':p.get('file_path',''),'pii_type':p.get('pii_type','OTHER'),'confirmed':None,'redacted':False}
        return f'Flagged {cid}: {p.get("pii_type","OTHER")} | {text!r}'

    def _t_advance_phase(self, p):
        if self.phase=='SCAN':
            if not self.candidates: raise ValueError('No candidates. Flag PII first.')
            self.phase='CLASSIFY'; return f'-> CLASSIFY. {len(self.candidates)} candidates.'
        elif self.phase=='CLASSIFY':
            unc=[c for c,v in self.candidates.items() if v['confirmed'] is None]
            if unc: raise ValueError(f'Classify all first. Unclassified: {unc}')
            self.phase='REDACT'; conf=[c for c,v in self.candidates.items() if v['confirmed']]
            return f'-> REDACT. {len(conf)} confirmed PII.'
        else: raise ValueError('Already in REDACT. Use redact_span + submit.')

    def _t_list_candidates(self, p):
        self._req('CLASSIFY','list_candidates')
        lines=[f'  {c}: [{"CONFIRMED" if v["confirmed"] else ("REJECTED" if v["confirmed"]==False else "PENDING")}] {v["pii_type"]} | {v["text"]!r}' for c,v in self.candidates.items()]
        return 'Candidates:\n'+('\n'.join(lines) or 'none')

    def _t_classify_candidate(self, p):
        self._req('CLASSIFY','classify_candidate'); cid=p.get('candidate_id','')
        if cid not in self.candidates: raise ValueError(f'Unknown {cid}.')
        c=p.get('confirmed')
        if c is None: raise ValueError('confirmed required.')
        self.candidates[cid]['confirmed']=bool(c)
        return f'{cid} {"CONFIRMED" if c else "REJECTED"}: {self.candidates[cid]["text"]!r}'

    def _t_redact_span(self, p):
        self._req('REDACT','redact_span'); cid=p.get('candidate_id','')
        if cid not in self.candidates: raise ValueError(f'Unknown {cid}.')
        c=self.candidates[cid]
        if not c['confirmed']: raise ValueError(f'{cid} not confirmed.')
        if c['redacted']: return f'{cid} already redacted.'
        text,fp = c['text'],c['file_path']
        if fp and fp in self.virtual_fs and text in self.virtual_fs[fp]:
            self.virtual_fs[fp]=self.virtual_fs[fp].replace(text,'[REDACTED]')
        else:
            for fname in self.virtual_fs:
                if text in self.virtual_fs[fname]: self.virtual_fs[fname]=self.virtual_fs[fname].replace(text,'[REDACTED]')
        c['redacted']=True; return f'Redacted {cid}: {text!r} -> [REDACTED]'

    def _t_submit(self, p):
        self._req('REDACT','submit'); self.reward,metrics=self._compute_reward(); self.done=True
        return f'Done. reward={self.reward:.4f} | {metrics}'

    def _compute_reward(self):
        total=len(self.pii_list)
        if total==0: return 0.5,{}
        flagged={v['text'] for v in self.candidates.values()}
        hits=sum(1 for p in self.pii_list if p in flagged or any(p in ft or ft in p for ft in flagged))
        scan_recall=hits/total
        conf=[v for v in self.candidates.values() if v['confirmed']]
        if not conf: precision=0.0
        else:
            tp=sum(1 for c in conf if any(c['text'] in p or p in c['text'] for p in self.pii_list))
            precision=tp/max(1,len(conf))
        all_content='\n'.join(self.virtual_fs.values())
        still=sum(1 for p in self.pii_list if p in all_content)
        redact_complete=1.0-(still/total)
        if scan_recall>=0.99 and precision>=0.99 and redact_complete>=0.99: r=1.0
        elif scan_recall>=0.5 and redact_complete>0: r=0.3+0.6*(scan_recall*precision*redact_complete)
        else: r=0.05
        return max(0.001,min(0.999,r)), {'scan_recall':round(scan_recall,3),'precision':round(precision,3),'redact_complete':round(redact_complete,3)}

    def _obs(self,reward,result,done=False):
        self.cumulative+=reward
        return {'phase':self.phase,'result':result,'reward':reward,'cumulative':round(self.cumulative,4),'done':done,'pii_count':len(self.pii_list)}

    def close(self): pass


# Sanity check
env = ComplianceGuardEnv()
env.reset(seed=42, level=1)
obs = env.step(json.dumps({'tool':'list_files'}))
assert 'server_logs.txt' in obs['result'], 'Env broken!'
print('ComplianceGuardEnv OK:', obs['result'])

## Cell 3 — Reward function (real episodes, no oracle)

Model outputs ALL actions in one completion. We execute them and return real episode reward.  
The unit test at the end MUST print `Reward function verified` before you run training.

In [ ]:
import re


def extract_json_actions(text: str) -> list:
    """Extract all JSON objects from model output (one per line)."""
    actions = []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith('{') and line.endswith('}'):
            actions.append(line)
        else:
            m = re.search(r'\{[^{}]+\}', line)
            if m:
                actions.append(m.group(0))
    return actions


def run_episode_from_actions(action_strings: list, level: int, seed: int) -> float:
    """
    Execute action list through fresh env. Return real episode reward.
    No oracle — model must do the work itself.
    """
    env = ComplianceGuardEnv()
    env.reset(seed=seed, level=level)

    if not action_strings:
        return 0.05

    for action_str in action_strings[:40]:  # cap at 40 steps
        try:
            obs = env.step(action_str)
        except Exception:
            continue
        if obs.get('done'):
            return float(env.reward)

    # Episode didn't reach submit — partial credit for progress
    if env.phase == 'REDACT':   return 0.40
    if env.phase == 'CLASSIFY': return 0.20
    if env.candidates:          return 0.10
    return 0.05


def compliance_reward(completions, prompts=None, **kwargs):
    """
    TRL reward_funcs: (completions, prompts, **dataset_columns).
    level/seed come from dataset columns passed as kwargs.
    Each completion = model's full action plan (one JSON per line).
    """
    levels = list(kwargs.get('level', []))
    seeds  = list(kwargs.get('seed',  []))
    rewards = []

    for i, comp in enumerate(completions):
        if isinstance(comp, list):
            text = comp[0].get('content', '') if comp else ''
        elif isinstance(comp, dict):
            text = comp.get('content', '')
        else:
            text = str(comp)

        level = int(levels[i]) if i < len(levels) else random.randint(1, 3)
        seed  = int(seeds[i])  if i < len(seeds)  else i * 7 + 42

        try:
            actions = extract_json_actions(text)
            r = run_episode_from_actions(actions, level=level, seed=seed)
        except Exception:
            r = 0.05

        rewards.append(float(r))

    return rewards


# ── Unit test ───────────────────────────────────────────────────────────────
# Build a perfect action sequence for seed=42 level=1
random.seed(42)
_e = ComplianceGuardEnv(); _e.reset(seed=42, level=1)
_pii = _e.pii_list[:]; _files = list(_e.virtual_fs.keys())
_acts_perfect = (
    [json.dumps({'tool':'list_files'}),
     json.dumps({'tool':'read_file','file_path':_files[0]})]
    + [json.dumps({'tool':'flag_candidate','text':p,'file_path':_files[0],'pii_type':'NAME'}) for p in _pii]
    + [json.dumps({'tool':'advance_phase'})]
    + [json.dumps({'tool':'classify_candidate','candidate_id':f'c{j}','confirmed':True}) for j in range(len(_pii))]
    + [json.dumps({'tool':'advance_phase'})]
    + [json.dumps({'tool':'redact_span','candidate_id':f'c{j}'}) for j in range(len(_pii))]
    + [json.dumps({'tool':'submit'})]
)

r_perfect = run_episode_from_actions(_acts_perfect, level=1, seed=42)
r_empty   = run_episode_from_actions([], level=1, seed=42)
r_partial = run_episode_from_actions([json.dumps({'tool':'list_files'})], level=1, seed=42)

print(f'Perfect episode: {r_perfect:.4f}  (expected ~1.0)')
print(f'Empty episode:   {r_empty:.4f}   (expected 0.05)')
print(f'Partial (1 act): {r_partial:.4f}   (expected 0.05)')
assert r_perfect > 0.9,  f'BROKEN: perfect should be >0.9, got {r_perfect}'
assert r_empty   == 0.05, f'BROKEN: empty should be 0.05, got {r_empty}'
print('Reward function verified — safe to run training')

## Cell 4 — System prompt and dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a PII compliance agent. Redact all personally identifiable information.

PII to flag and redact: person names, email addresses, phone numbers, SSNs.
NOT PII: dates (2026-01-01), IP addresses (192.168.1.1), system emails (test@example.com), log levels.

3-phase workflow: SCAN -> CLASSIFY -> REDACT

SCAN tools:
  {"tool": "list_files"}
  {"tool": "read_file", "file_path": "<name>"}
  {"tool": "flag_candidate", "text": "<pii_text>", "file_path": "<name>", "pii_type": "<NAME|EMAIL|PHONE|SSN>"}
  {"tool": "advance_phase"}
CLASSIFY tools:
  {"tool": "list_candidates"}
  {"tool": "classify_candidate", "candidate_id": "<id>", "confirmed": true}
  {"tool": "advance_phase"}
REDACT tools:
  {"tool": "redact_span", "candidate_id": "<id>"}
  {"tool": "submit"}

Output ALL actions to complete the full episode, ONE JSON per line. No prose."""


def make_initial_obs(level: int, seed: int) -> str:
    env = ComplianceGuardEnv()
    return env.reset(seed=seed, level=level)


def build_prompt(level: int, seed: int) -> str:
    return f'{SYSTEM_PROMPT}\n\nInitial state: {make_initial_obs(level, seed)}\n\nActions:'


def build_dataset(n_per_level: int = 50) -> Dataset:
    rows = []
    for level in [1, 1, 1, 2, 2, 3]:  # weighted toward easy
        for seed in range(n_per_level):
            s = seed + level * 1000
            rows.append({'prompt': build_prompt(level, s), 'level': level, 'seed': s})
    random.shuffle(rows)
    return Dataset.from_list(rows)


dataset = build_dataset(n_per_level=50)
print(f'Dataset: {len(dataset)} rows | levels: {sorted(set(dataset["level"]))}')
print('Sample (first 300 chars):')
print(dataset[0]['prompt'][:300])

## Cell 5 — Load model (T4-safe)

In [ ]:
import torch
from unsloth import FastLanguageModel, PatchFastRL
from trl import GRPOConfig, GRPOTrainer

# Must patch BEFORE using GRPOTrainer
PatchFastRL('GRPO', FastLanguageModel)

model, tokenizer = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-1.5B-Instruct',
    max_seq_length=512,
    load_in_4bit=True,
    fast_inference=False,   # required on T4 — no vLLM
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Model: Qwen2.5-1.5B-Instruct 4-bit QLoRA')
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Cell 6 — GRPO Training

Uses `reward_funcs=compliance_reward` (T4-compatible, no vLLM).  
**After step 5, check `reward_std` — it must be > 0.05. If it is 0.000, stop and fix the reward function.**

In [ ]:
config = GRPOConfig(
    output_dir='checkpoints/grpo',
    max_steps=200,
    num_generations=4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    use_vllm=False,               # required on T4
    max_completion_length=512,
    logging_steps=1,
    save_steps=50,
    report_to='none',
)

# NOTE: use args= not config= (Unsloth patched GRPOTrainer renamed the param)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=config,
    train_dataset=dataset,
    reward_funcs=compliance_reward,
)

print('Starting GRPO training...')
print('HEALTH CHECK: reward_std should be > 0.05 after step 5.')
print('If reward=0.999 and reward_std=0.000 from step 1 -> training is broken (oracle hack).')
trainer.train()
print('Training complete!')

# Save reward log
import csv, pathlib
log = trainer.state.log_history

def _get_reward(e):
    return e.get('reward', e.get('train/reward', e.get('rewards')))

reward_rows = [{'step': e['step'], 'reward': _get_reward(e)} for e in log if _get_reward(e) is not None]
if reward_rows:
    with open('reward_log.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['step','reward'])
        w.writeheader(); w.writerows(reward_rows)
    vals = [r['reward'] for r in reward_rows]
    print(f'reward_log.csv saved ({len(vals)} rows)')
    print(f'  step-1 reward: {vals[0]:.4f}')
    print(f'  last-10 avg:   {sum(vals[-10:])/len(vals[-10:]):.4f}')
    print(f'  first-10 avg:  {sum(vals[:10])/10:.4f}')

## Cell 7 — Save model

In [ ]:
model.save_pretrained('checkpoints/grpo/final-adapter')
tokenizer.save_pretrained('checkpoints/grpo/final-adapter')
print('Adapter saved: checkpoints/grpo/final-adapter')

model.save_pretrained_merged('checkpoints/grpo/final-merged', tokenizer, save_method='merged_16bit')
print('Merged 16-bit saved: checkpoints/grpo/final-merged')

## Cell 8 — Plot reward curve

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np, csv

with open('reward_log.csv') as f:
    rows = list(csv.DictReader(f))
steps   = [int(r['step'])    for r in rows]
rewards = [float(r['reward']) for r in rows]
print(f'Loaded {len(rewards)} entries')

def smooth(v, w=10): return np.convolve(v, np.ones(w)/w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(14,5))
fig.suptitle('ComplianceGuard GRPO Training', fontsize=14, fontweight='bold')

axes[0].plot(steps, rewards, alpha=0.25, color='steelblue', lw=0.8, label='raw')
if len(rewards)>=10:
    sw=smooth(rewards)
    axes[0].plot(steps[:len(sw)], sw, color='steelblue', lw=2, label='smoothed (w=10)')
axes[0].axhline(0.70, color='green', ls='--', lw=1.5, label='success threshold (0.70)')
axes[0].set(xlabel='Step', ylabel='Reward', title='Episode Reward'); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(-0.05,1.1)

cm = np.cumsum(rewards) / (np.arange(len(rewards))+1)
axes[1].plot(steps, cm, color='darkorange', lw=2, label='cumulative mean')
axes[1].axhline(0.70, color='green', ls='--', lw=1.5, label='success threshold')
axes[1].set(xlabel='Step', ylabel='Cumulative mean reward', title='Cumulative Mean'); axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_ylim(0,1.05)

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150, bbox_inches='tight')
print('Saved reward_curve.png')

## Cell 9 — Download results

In [ ]:
from google.colab import files
files.download('reward_log.csv')
files.download('reward_curve.png')
print('Downloaded. Place reward_log.csv in repo root.')

## Troubleshooting

| Error | Fix |
|-------|-----|
| `reward=0.999, reward_std=0.000` from step 1 | Oracle hacking — are you using THIS notebook? Check Cell 3 unit test passes. |
| `ImportError: vLLM` | `fast_inference=False` in Cell 5 |
| `TypeError: got unexpected kwarg 'config'` | Use `args=config` not `config=config` in Cell 6 |
| `AssertionError: environment_factory` | Unsloth removed it — use `reward_funcs=` (already done here) |
| CUDA OOM | Set `per_device_train_batch_size=1`, `max_completion_length=256` |